# OpenAlex

Open catalog of the global research system: authors, works, institutions, funders, journals, topics.  Free, no API key — just a contact email in the User-Agent for polite-pool access.

**Base URL:** `https://api.openalex.org`  
**PSU ROR:** `https://ror.org/04p491231`

What it gives the pipeline: the **researcher's academic record** — works count, citation metrics, h-index, topics, affiliations history, and (the key new addition) the **complete DOI list** we feed to Overton.

## Setup

In [ ]:
import json, os, time
import requests

OPENALEX_EMAIL = os.environ.get('OPENALEX_EMAIL', 'overton-pipeline@psu.edu')
PSU_ROR = 'https://ror.org/04p491231'
OA_BASE = 'https://api.openalex.org'

def oa_get(path, params=None):
    h = {'User-Agent': f'mailto:{OPENALEX_EMAIL}'}
    r = requests.get(f'{OA_BASE}{path}', params=params, headers=h, timeout=30)
    r.raise_for_status()
    time.sleep(0.1)
    return r.json()

def pretty(o, limit=2000):
    s = json.dumps(o, indent=2, default=str, ensure_ascii=False)
    print(s if len(s) < limit else s[:limit] + f'\n... ({len(s):,} chars total)')

print('ready')

## 1. `/authors/orcid:<ORCID>` — author by ORCID (the happy path)

When we have an ORCID, this single call gives us everything: name, current institution, works count, citation metrics, top topics, and a `works_api_url` for fetching their full publication list.

In [ ]:
# Pick a researcher we know is mapped
with open('../data/pipeline/orcid_webaccess_map.json') as f:
    orcid_map = json.load(f)
sample_orcid = next(iter(orcid_map))

author = oa_get(f'/authors/orcid:{sample_orcid}')
print(f'Top-level author keys: {list(author.keys())}\n')
print('Curated subset:')
for k in ['id', 'orcid', 'display_name', 'works_count', 'cited_by_count']:
    print(f'  {k:<25} {author.get(k)!r}')
print(f'  h_index                   {author.get("summary_stats", {}).get("h_index")}')
print(f'  i10_index                 {author.get("summary_stats", {}).get("i10_index")}')
print(f'  2yr_mean_citedness        {author.get("summary_stats", {}).get("2yr_mean_citedness")}')
print(f'  works_api_url             {author.get("works_api_url")}')

## 2. Author's affiliations history
Every institution they've published from, with year ranges.  Useful for distinguishing 'currently at PSU' from 'left PSU but kept publishing'.

In [ ]:
for aff in author.get('affiliations', [])[:6]:
    inst = aff.get('institution', {})
    years = aff.get('years', [])
    yr_range = f'{min(years)}-{max(years)}' if years else '?'
    print(f'  {inst.get("display_name"):<55} {inst.get("country_code")}  {yr_range}')
    print(f'    ror: {inst.get("ror")}')

print('\nlast_known_institutions (most recent):')
for i in author.get('last_known_institutions', [])[:3]:
    print(f'  {i.get("display_name")}  ({i.get("ror")})')

## 3. Topics — 4-level subject hierarchy
Each topic has a subfield → field → domain hierarchy, and a count of how many of the author's works fall under it.

In [ ]:
for t in author.get('topics', [])[:5]:
    print(f'  {t["display_name"]:<55} works={t.get("count"):<3}')
    print(f'    {t["subfield"]["display_name"]} > {t["field"]["display_name"]} > {t["domain"]["display_name"]}')

## 4. Counts by year
Year-over-year publication and citation trajectory.  Useful for trend visualizations.

In [ ]:
for c in author.get('counts_by_year', [])[:8]:
    print(f'  {c["year"]}  works={c["works_count"]:<3}  citations={c["cited_by_count"]:<5}  oa_works={c.get("oa_works_count",0)}')

## 5. `/authors?search=<name>&filter=ror:<PSU>` — name search (the no-ORCID path)

When we don't have an ORCID, search by name + PSU affiliation.  OpenAlex handles unicode/accents/middle-initial variations and ranks by relevance.

We can include or exclude `has_orcid:true` — if we want every PSU author (ORCID or not, for the DOI-bridge approach), drop the filter.

In [ ]:
resp = oa_get('/authors', params={
    'filter': f'affiliations.institution.ror:{PSU_ROR}',
    'search': 'Anthony Pegg',  # an unmapped researcher with no ORCID on OpenAlex
    'per-page': 5,
    'select': 'id,orcid,display_name,works_count,last_known_institutions',
})
print(f'Total matches: {resp["meta"]["count"]}\n')
for h in resp['results']:
    inst = (h.get('last_known_institutions') or [{}])[0].get('display_name', '-')
    orcid = h.get('orcid') or '(none)'
    print(f'  {orcid:<45} {h["display_name"]:<35}  works={h["works_count"]:<4}  inst={inst}')

## 6. `/works?filter=author.orcid:<ORCID>` — full works list

This is the **key call for the new DOI-driven Overton flow.** Paginated via `cursor` (use `*` to start).  At `per-page=200` the request returns 200 works with their DOIs in one call.

In [ ]:
# First page only - just to see the shape
works = oa_get('/works', params={
    'filter': f'author.orcid:{sample_orcid}',
    'per-page': 5,
    'cursor': '*',
    'select': 'id,doi,title,publication_year,type,open_access,authorships',
})
print(f'Total works for this researcher: {works["meta"]["count"]}\n')
for w in works['results']:
    print(f'  {w.get("doi") or "(no doi)":<45} {w.get("publication_year")} [{w.get("type")}] {w.get("title","")[:60]}')

In [ ]:
# Inspect a single work's full structure
first_work = works['results'][0]
print(f'Work keys: {list(first_work.keys())}\n')
print('Authorships (first 3):')
for a in first_work.get('authorships', [])[:3]:
    au = a.get('author', {})
    print(f'  position={a.get("author_position"):<6} {au.get("display_name"):<30} '
          f'orcid={au.get("orcid") or "(none)":<40}')
print(f'\nopen_access: {first_work.get("open_access")}')

## 7. Full DOI harvest — pulling every DOI for one researcher

This is what feeds the new Overton stage.  Loops `cursor` pagination until exhausted.

In [ ]:
dois = []
cursor = '*'
pages = 0
while cursor and pages < 10:  # safety cap
    pages += 1
    d = oa_get('/works', params={
        'filter': f'author.orcid:{sample_orcid}',
        'per-page': 200,
        'cursor': cursor,
        'select': 'doi',
    })
    for w in d.get('results', []):
        if w.get('doi'):
            dois.append(w['doi'].replace('https://doi.org/', ''))
    cursor = d.get('meta', {}).get('next_cursor')

print(f'Pulled {len(dois)} DOIs across {pages} pages')
print(f'Sample: {dois[:5]}')

## 8. `/works/doi:<DOI>` — single work lookup by DOI
Useful when you want to inspect one work in detail (full authorships with ORCIDs, OA links, references).

In [ ]:
work = oa_get(f'/works/doi:{dois[0]}')
print(f'Title: {work.get("title")[:80]}')
print(f'Year:  {work.get("publication_year")}')
print(f'OA:    {work.get("open_access")}')
print(f'Source: {(work.get("primary_location") or {}).get("source", {}).get("display_name")}')
print(f'\nAuthorships:')
for a in work.get('authorships', [])[:6]:
    au = a.get('author', {})
    print(f'  {au.get("orcid") or "(no orcid)":<45} {au.get("display_name")}')

## 9. `/funders/<id>` — funder records
Cross-link with RMD grant agencies for funder-level impact metrics.

In [ ]:
# Search for a funder by name
funders = oa_get('/funders', params={'search': 'NIH National Institutes of Health', 'per-page': 3, 'select': 'id,display_name,country_code,works_count,cited_by_count,ids'})
for f in funders['results']:
    print(f'  {f["display_name"]:<50}  {f["country_code"]}  works={f["works_count"]}  cited={f["cited_by_count"]}')
    print(f'    ror: {f["ids"].get("ror")}')

## 10. `/topics/<id>` — topic record (subfield/field/domain)
Useful for showing topic context — keywords, sibling topics, global stats.

In [ ]:
if author.get('topics'):
    topic_id = author['topics'][0]['id'].replace('https://openalex.org/', '')
    t = oa_get(f'/topics/{topic_id}')
    print(f'  Topic: {t["display_name"]}')
    print(f'  Subfield/Field/Domain: {t["subfield"]["display_name"]} > {t["field"]["display_name"]} > {t["domain"]["display_name"]}')
    print(f'  Global works: {t["works_count"]}, citations: {t["cited_by_count"]}')
    print(f'  Keywords: {t.get("keywords",[])[:5]}')

## What the pipeline pulls today (Stage 2: OpenAlex)

Per researcher (looped over the cohort from Stage 1):

| Endpoint | Used for |
|---|---|
| `/authors/orcid:<orcid>` | author profile + summary stats |

After the planned refactor (Option B), Stage 2 will additionally pull:

| Endpoint | Used for |
|---|---|
| `/authors?search=<name>&filter=ror:PSU` | name-search fallback for the 235 researchers without ORCIDs in RMD |
| `/works?filter=author.orcid:<x>` (or `author.id:<x>`) | full DOI list per researcher → fed into Overton DOI-set |